# Proteomics Contaminant Filtering & Visualization

**How to use this notebook:**
1. Edit **Section 1** to set your experiment subfolder and input filename — this is the only cell you need to change between experiments
2. Edit **Section 2** if you want to add/remove genes from the contaminant lists (shared across all experiments)
3. Run all cells — outputs are saved automatically to your experiment subfolder

**Folder structure expected:**
```
~/
    contaminant_filter.ipynb        ← this notebook
    uniprotkb_...tsv                ← UniProt annotation file (shared)
    PAG/
        your_proteomics_data.xlsx   ← input
        outputs/                    ← created automatically
    Nrxn/
        your_proteomics_data.xlsx
        outputs/
    ...
```

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import os
warnings.filterwarnings('ignore')

print('Libraries loaded.')

## 1. ⚙️ EXPERIMENT SETTINGS — Edit This Cell Each Time
Everything else runs automatically from these settings.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  CHANGE THESE FOR EACH EXPERIMENT
# ══════════════════════════════════════════════════════════════════════════════

# Root folder where this notebook and the UniProt file live
ROOT_DIR     = '~/'

# Subfolder for this experiment (e.g. 'PAG', 'Nrxn', 'PFC')
EXPERIMENT   = 'Cocaine_vs_nobiotin'

# Filename of your proteomics Excel file inside the experiment subfolder
INPUT_FILE   = 'pfccocaine_vs_nobiotin_all_nofilters_nonorm_2peptides.xlsx'

# UniProt TSV filename (lives in ROOT_DIR, shared across experiments)
UNIPROT_FILE = 'uniprotkb_organism_id_10090_AND_reviewe_2026_06_16.tsv'

# ══════════════════════════════════════════════════════════════════════════════
#  COLUMN NAMES — only change if your file headers differ
# ══════════════════════════════════════════════════════════════════════════════
COL_GENES       = 'Genes'
COL_LOG2FC      = 'AVG Log2 Ratio'
COL_PVALUE      = 'Pvalue'
COL_PROTEINS    = 'ProteinGroups'
COL_DESCRIPTION = 'ProteinDescriptions'

# ══════════════════════════════════════════════════════════════════════════════
#  PATHS — auto-constructed, no need to edit
# ══════════════════════════════════════════════════════════════════════════════
EXPERIMENT_DIR   = os.path.join(ROOT_DIR, EXPERIMENT)
OUTPUT_DIR       = os.path.join(EXPERIMENT_DIR, 'outputs')
PROTEOMICS_PATH  = os.path.join(EXPERIMENT_DIR, INPUT_FILE)
UNIPROT_PATH     = os.path.join(ROOT_DIR, UNIPROT_FILE)
OUTPUT_BASE      = os.path.join(OUTPUT_DIR, f'{EXPERIMENT}_contaminants')

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Experiment  : {EXPERIMENT}')
print(f'Input file  : {PROTEOMICS_PATH}')
print(f'UniProt file: {UNIPROT_PATH}')
print(f'Output dir  : {OUTPUT_DIR}')
print()

# Sanity checks
for path, label in [(PROTEOMICS_PATH, 'Proteomics file'),
                    (UNIPROT_PATH,    'UniProt file')]:
    if os.path.exists(path):
        print(f'  ✓ {label} found')
    else:
        print(f'  ✗ {label} NOT FOUND: {path}')

## 2. 🧹 Contaminant Lists — Edit to Add/Remove Genes
These are shared across all experiments. Add gene symbols to any category, or add a new category dict entry.

In [ ]:
# ── GENE-NAME BASED HARD FILTERS ─────────────────────────────────────────────
# Case-insensitive. Semicolon-separated multi-gene cells are split automatically.
# Flag the whole protein group if ANY gene in the group matches.

CONTAMINANT_GENES = {

    # Keratins (skin/epithelial intermediate filaments)
    'Keratins': [
        'Krt1','Krt2','Krt3','Krt4','Krt5','Krt6a','Krt6b','Krt7','Krt8',
        'Krt9','Krt10','Krt12','Krt13','Krt14','Krt15','Krt16','Krt17',
        'Krt18','Krt19','Krt20','Krt23','Krt24','Krt25','Krt26','Krt27',
        'Krt28','Krt31','Krt32','Krt33a','Krt33b','Krt34','Krt35','Krt36',
        'Krt37','Krt38','Krt39','Krt40','Krt71','Krt72','Krt73','Krt74',
        'Krt75','Krt76','Krt77','Krt78','Krt79','Krt80','Krt81','Krt82',
        'Krt83','Krt84','Krt85','Krt86',
    ],

    # Desmosomal complex (epithelial cell-cell junctions / ependymal)
    'Desmosomal': [
        'Dsg1','Dsg2','Dsg3','Dsg4',           # Desmogleins
        'Dsc1','Dsc2','Dsc3',                   # Desmocollins
        'Dsp',                                  # Desmoplakin
        'Pkp1','Pkp2','Pkp3','Pkp4',            # Plakophilins
        'Jup',                                  # Plakoglobin
        'Evpl',                                 # Envoplakin
        'Ppl',                                  # Periplakin
    ],

    # Cornified envelope / skin differentiation
    'Cornified_envelope': [
        'Tgm1','Tgm3','Tgm5',                  # Transglutaminases
        'Flg','Flg2',                           # Filaggrin
        'Lor',                                  # Loricrin
        'Ivl',                                  # Involucrin
        'Sprr1a','Sprr1b','Sprr2a','Sprr2b',    # Small proline-rich proteins
        'Lce1a','Lce1b','Lce1c',               # Late cornified envelope proteins
    ],

    # Blood plasma proteins
    'Blood_plasma': [
        'Alb',                                  # Albumin
        'Fga','Fgb','Fgg',                      # Fibrinogen chains
        'C3','C4a','C4b','C1qa','C1qb','C1qc',  # Complement
        'A2m',                                  # Alpha-2-macroglobulin
        'Tf',                                   # Transferrin
        'Hp',                                   # Haptoglobin
        'Vtn',                                  # Vitronectin
        'Fn1',                                  # Fibronectin
        'Apoa1','Apoa2','Apob','Apoe',          # Apolipoproteins
        'Fetub','Ahsg',                         # Fetuin
        'Serpina1','Serpina3',                  # Serpins
    ],

    # Red blood cell proteins
    'Red_blood_cell': [
        'Hba','Hba-a1','Hba-a2',               # Hemoglobin alpha
        'Hbb','Hbb-b1','Hbb-b2','Hbb-bs',      # Hemoglobin beta
        'Epb42',                                # Dematin (Band 4.2)
        'Epb41',                                # Band 4.1
        'Ank1',                                 # Ankyrin-1
        'Spta1','Sptb',                         # Spectrin alpha/beta erythrocytic
        'Slc4a1',                               # Band 3 / anion exchanger
        'Gypa','Gypb',                          # Glycophorin
        'Ca1',                                  # Carbonic anhydrase 1
        'Prdx2',                                # Peroxiredoxin-2 (highly abundant in RBCs)
    ],

    # Platelet proteins
    'Platelets': [
        'Itga2b','Itgb3',                       # Integrin alphaIIb/beta3
        'Vwf',                                  # Von Willebrand factor
        'Thbs1',                                # Thrombospondin-1
        'Pf4',                                  # Platelet factor 4
    ],

    # Immunoglobulins (blood / immune contamination)
    # Note: Igkv*/Ighv* prefixes are also caught automatically below
    'Immunoglobulins': [
        'Ighg1','Ighg2b','Ighg2c','Ighg3',
        'Igha','Ighm','Ighd',
        'Igh-1a','Igh-1b','Igh-2','Igh-3',
        'Igkc','Iglc1','Iglc2','Iglc3',
    ],

    # Structural collagens (vascular / meningeal / ependymal ECM)
    'Collagens': [
        'Col1a1','Col1a2','Col2a1','Col3a1',
        'Col4a1','Col4a2','Col4a3','Col4a4','Col4a5','Col4a6',
        'Col5a1','Col5a2','Col5a3',
        'Col6a1','Col6a2','Col6a3',
        'Col12a1','Col14a1','Col18a1',
    ],

    # Laminin subunits (basement membrane)
    'Laminins': [
        'Lama1','Lama2','Lama4','Lama5',
        'Lamb1','Lamb2',
        'Lamc1','Lamc2','Lamc3',
    ],

    # Other ECM / vascular contaminants
    'Other_ECM': [
        'Fbln1','Fbln2','Fbln5',               # Fibulins
        'Eln',                                  # Elastin
        'Mfap2','Mfap4','Mfap5',               # Microfibrillar proteins
        'Ltbp1','Ltbp2','Ltbp3','Ltbp4',       # Latent TGF-beta binding proteins
        'Tgfbi',                                # TGF-beta induced protein
    ],

    # Lab reagent proteins
    'Lab_reagents': [
        'Try1','Prss1','Prss2',                 # Trypsin
        'Lys',                                  # Lysozyme
    ],
}

# ── UNIPROT SUBCELLULAR LOCATION KEYWORDS ────────────────────────────────────
# Any protein whose UniProt 'Subcellular location' field contains one of these
# strings (case-insensitive) will be flagged.
CONTAMINANT_LOCATION_KEYWORDS = [
    'cornified envelope',
    'extracellular matrix',
    'basement membrane',
    'desmosome',
]

# ─────────────────────────────────────────────────────────────────────────────
total_genes = sum(len(v) for v in CONTAMINANT_GENES.values())
print(f'Contaminant gene list: {len(CONTAMINANT_GENES)} categories, {total_genes} gene symbols')
print(f'UniProt location keywords: {len(CONTAMINANT_LOCATION_KEYWORDS)}')

## 3. Load Data

In [ ]:
prot = pd.read_excel(PROTEOMICS_PATH)
print(f'Proteomics data: {prot.shape[0]} protein groups')

# Auto-detect Pvalue vs Qvalue column
if COL_PVALUE not in prot.columns:
    for candidate in ('Qvalue', 'Pvalue', 'pvalue', 'qvalue', 'p_value', 'q_value'):
        if candidate in prot.columns:
            print(f'  ℹ "{COL_PVALUE}" not found — using "{candidate}" instead')
            COL_PVALUE = candidate
            break
    else:
        print(f'  ✗ Neither Pvalue nor Qvalue column found — check COL_PVALUE setting')

print(f'  Using p/q-value column: {COL_PVALUE}')
prot.head(3)

In [ ]:
uniprot = pd.read_csv(UNIPROT_PATH, sep='\t', low_memory=False)
print(f'UniProt: {uniprot.shape[0]} entries')

# Build gene → UniProt row lookup (lowercase, space-separated UniProt gene names)
gene_to_uniprot = {}
for idx, row in uniprot.iterrows():
    gnames = str(row.get('Gene Names', ''))
    if gnames and gnames != 'nan':
        for g in gnames.split():
            gene_to_uniprot.setdefault(g.strip().lower(), []).append(idx)

print(f'UniProt gene lookup: {len(gene_to_uniprot)} unique symbols')

## 4. Flag Contaminants

In [ ]:
# Flat lowercase lookup: gene_symbol -> category
all_contaminant_genes = {}
for category, genes in CONTAMINANT_GENES.items():
    for g in genes:
        all_contaminant_genes[g.lower()] = category

def get_gene_list(cell_value):
    """Split semicolon-separated gene cell into lowercase list."""
    if pd.isna(cell_value) or str(cell_value).strip() in ('', 'nan', 'NaN'):
        return []
    return [g.strip().lower() for g in str(cell_value).split(';') if g.strip()]

def check_uniprot_location(gene_list):
    """Return (flagged, reason) based on UniProt subcellular location keywords."""
    for gene in gene_list:
        for idx in gene_to_uniprot.get(gene, []):
            loc = str(uniprot.loc[idx, 'Subcellular location [CC]']).lower()
            for kw in CONTAMINANT_LOCATION_KEYWORDS:
                if kw.lower() in loc:
                    return True, f'UniProt location: {kw}'
    return False, ''

flags, categories, reasons, matched_genes = [], [], [], []

for _, row in prot.iterrows():
    genes    = get_gene_list(row[COL_GENES])
    flagged  = False
    category = ''
    reason   = ''
    matched  = []

    # Check 1: explicit gene-name list
    for gene in genes:
        if gene in all_contaminant_genes:
            flagged  = True
            category = all_contaminant_genes[gene]
            reason   = f'Gene name match: {gene}'
            matched.append(gene)

    # Check 2: Ig variable region prefix catch (Igkv*, Ighv*, Iglv*)
    if not flagged:
        for gene in genes:
            if any(gene.startswith(p) for p in
                   ('ighv','igkv','iglv','ighg','igha','ighm')):
                flagged  = True
                category = 'Immunoglobulins'
                reason   = f'Ig prefix: {gene}'
                matched.append(gene)

    # Check 3: UniProt subcellular location
    if not flagged:
        loc_hit, loc_reason = check_uniprot_location(genes)
        if loc_hit:
            flagged  = True
            category = 'UniProt_location'
            reason   = loc_reason

    flags.append(flagged)
    categories.append(category if flagged else '')
    reasons.append(reason if flagged else '')
    matched_genes.append('; '.join(matched) if matched else '')

prot['is_contaminant']       = flags
prot['contaminant_category'] = categories
prot['contaminant_reason']   = reasons
prot['contaminant_genes']    = matched_genes

n_contam = prot['is_contaminant'].sum()
n_clean  = (~prot['is_contaminant']).sum()
print(f'Total protein groups : {len(prot)}')
print(f'Contaminants flagged : {n_contam} ({100*n_contam/len(prot):.1f}%)')
print(f'Clean proteins       : {n_clean}  ({100*n_clean/len(prot):.1f}%)')

In [ ]:
# Breakdown by category
cat_counts = (prot[prot['is_contaminant']]
              .groupby('contaminant_category').size()
              .sort_values(ascending=False))
print(f'\nContaminants by category ({EXPERIMENT}):')
print(cat_counts.to_string())

In [ ]:
# Table of flagged proteins
flagged_df = (prot[prot['is_contaminant']]
              [[COL_GENES, COL_LOG2FC, COL_PVALUE, COL_DESCRIPTION,
                'contaminant_category', 'contaminant_reason']]
              .sort_values('contaminant_category'))
print(f'Flagged contaminants ({len(flagged_df)}):')
flagged_df

## 5. Plots

In [ ]:
clean_df  = prot[~prot['is_contaminant']].dropna(subset=[COL_LOG2FC, COL_PVALUE])
contam_df = prot[ prot['is_contaminant']].dropna(subset=[COL_LOG2FC, COL_PVALUE])

C_CLEAN  = '#2196F3'
C_CONTAM = '#F44336'
ALPHA    = 0.6

# ── Distributions ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'[{EXPERIMENT}] Contaminant vs Clean Protein Distributions',
             fontsize=14, fontweight='bold', y=1.01)

ax = axes[0]
bins = np.linspace(min(prot[COL_LOG2FC].min(), -5), max(prot[COL_LOG2FC].max(), 5), 40)
ax.hist(clean_df[COL_LOG2FC],  bins=bins, color=C_CLEAN,  alpha=ALPHA,
        label=f'Clean (n={len(clean_df)})')
ax.hist(contam_df[COL_LOG2FC], bins=bins, color=C_CONTAM, alpha=ALPHA,
        label=f'Contaminants (n={len(contam_df)})')
ax.axvline(0,    color='black', linestyle='--', linewidth=1, alpha=0.5)
ax.axvline(0.58, color='gray',  linestyle=':',  linewidth=1.5, label='log2FC = 0.58')
ax.set_xlabel('AVG Log2 Ratio (+biotin / −biotin)', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Log2 Fold Change Distribution', fontsize=12)
ax.legend(fontsize=10)
ax.spines[['top','right']].set_visible(False)

ax = axes[1]
clean_logp  = -np.log10(clean_df[COL_PVALUE].clip(lower=1e-300))
contam_logp = -np.log10(contam_df[COL_PVALUE].clip(lower=1e-300))
bins_p = np.linspace(0, max(clean_logp.max(), contam_logp.max()) + 0.5, 40)
ax.hist(clean_logp,  bins=bins_p, color=C_CLEAN,  alpha=ALPHA,
        label=f'Clean (n={len(clean_df)})')
ax.hist(contam_logp, bins=bins_p, color=C_CONTAM, alpha=ALPHA,
        label=f'Contaminants (n={len(contam_df)})')
ax.axvline(-np.log10(0.05), color='gray', linestyle=':', linewidth=1.5,
           label=f'{COL_PVALUE} = 0.05')
ax.set_xlabel(f'−log10({COL_PVALUE})', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title(f'{COL_PVALUE} Distribution', fontsize=12)
ax.legend(fontsize=10)
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
out = f'{OUTPUT_BASE}_distributions.pdf'
plt.savefig(out, bbox_inches='tight', dpi=150)
plt.show()
print(f'Saved: {out}')

In [ ]:
# ── Volcano ───────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))

ax.scatter(clean_df[COL_LOG2FC],
           -np.log10(clean_df[COL_PVALUE].clip(lower=1e-300)),
           color=C_CLEAN, alpha=0.5, s=18,
           label=f'Clean (n={len(clean_df)})', zorder=2)
ax.scatter(contam_df[COL_LOG2FC],
           -np.log10(contam_df[COL_PVALUE].clip(lower=1e-300)),
           color=C_CONTAM, alpha=0.8, s=25,
           label=f'Contaminants (n={len(contam_df)})', zorder=3)

for _, row in contam_df.iterrows():
    gene_label = str(row[COL_GENES]).split(';')[0].strip()
    ax.annotate(gene_label,
                xy=(row[COL_LOG2FC], -np.log10(max(row[COL_PVALUE], 1e-300))),
                fontsize=6, color='#B71C1C',
                xytext=(3, 3), textcoords='offset points')

ax.axvline( 0.58, color='gray', linestyle=':', linewidth=1.5, label='log2FC = ±0.58')
ax.axvline(-0.58, color='gray', linestyle=':', linewidth=1.5)
ax.axhline(-np.log10(0.05), color='lightgray', linestyle='--', linewidth=1)
ax.set_xlabel('AVG Log2 Ratio (+biotin / −biotin)', fontsize=12)
ax.set_ylabel(f'−log10({COL_PVALUE})', fontsize=12)
ax.set_title(f'[{EXPERIMENT}] Volcano — Contaminants Highlighted',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
out = f'{OUTPUT_BASE}_volcano.pdf'
plt.savefig(out, bbox_inches='tight', dpi=150)
plt.show()
print(f'Saved: {out}')

In [ ]:
# ── Category bar chart ────────────────────────────────────────────────────────
if len(cat_counts) > 0:
    fig, ax = plt.subplots(figsize=(8, 4))
    cat_counts.plot(kind='barh', ax=ax, color='#EF5350', edgecolor='white')
    ax.set_xlabel('Number of Protein Groups', fontsize=11)
    ax.set_title(f'[{EXPERIMENT}] Contaminants by Category',
                 fontsize=13, fontweight='bold')
    ax.spines[['top','right']].set_visible(False)
    plt.tight_layout()
    out = f'{OUTPUT_BASE}_categories.pdf'
    plt.savefig(out, bbox_inches='tight', dpi=150)
    plt.show()
    print(f'Saved: {out}')

## 6. Export Results

In [ ]:
# Full dataset with contaminant flags
out_full = f'{OUTPUT_BASE}_flagged.xlsx'
prot.to_excel(out_full, index=False)
print(f'Full flagged dataset  → {out_full}')

# Clean proteins only
out_clean = f'{OUTPUT_BASE}_clean_only.xlsx'
prot[~prot['is_contaminant']].to_excel(out_clean, index=False)
print(f'Clean proteins only   → {out_clean}')

# Contaminants only
out_contam = f'{OUTPUT_BASE}_contaminants_only.xlsx'
prot[prot['is_contaminant']].to_excel(out_contam, index=False)
print(f'Contaminants only     → {out_contam}')

print(f'\n── Summary: {EXPERIMENT} ──────────────────────────')
print(f'  Input protein groups : {len(prot)}')
print(f'  Contaminants removed : {prot["is_contaminant"].sum()}')
print(f'  Clean proteins       : {(~prot["is_contaminant"]).sum()}')
print(f'  Output folder        : {OUTPUT_DIR}')